# 00 -- Load & Assemble the Freddie Mac data

**What this notebook does (plain English):** The raw Freddie Mac files have no
column headings and split each loan across two files -- one row of facts at the
loan's start, and one row *per month* of its life (millions of rows). This
notebook attaches the official column names, checks the layout is right, finds
whether and when each loan defaulted, and boils everything down to **one tidy
row per loan**. It then stacks **17 origination years (2006-2022)** together --
spanning the housing boom, the **2007-2009 financial crisis**, the recovery, the
long expansion, and the **2020 COVID** shock.

**Headline result:** the crisis and COVID vintages default far more often than
the calm expansion years -- a full cycle of good and bad years, which is what the
long-run regulatory calibration needs.

In [1]:
import sys, os
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, 'src')):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings('ignore')
print('project root:', ROOT)

project root: D:\Jane\Job Search\Github\bank\github project\freddie mac mortgage


In [2]:
# Read all 17 vintages (2006-2022), apply the 32-column layout, and collapse
# the monthly performance files down to one row per loan (this is the heavy step,
# ~minutes: tens of millions of servicing rows across the panel).
from src import loaders
df = loaders.load_all_vintages('data/raw data')
print('assembled loan-level table:', df.shape)

assembled loan-level table: (850000, 57)


In [3]:
# Cache the assembled table so every later notebook loads in seconds.
os.makedirs('data/processed', exist_ok=True)
df.to_parquet('data/processed/loan_level.parquet')
print('cached -> data/processed/loan_level.parquet')

cached -> data/processed/loan_level.parquet


In [4]:
# Build a data-quality + SEASONING summary per vintage (R3-D2). The one-year PD
# target needs >=12 months of performance to be observable, so we evidence how far
# each vintage is observed rather than assume it.
df['_seasoned_12m'] = df['months_observed'].fillna(0) >= 12
dq = df.groupby('vintage_year').agg(
    loans=('loan_sequence_number', 'size'),
    defaults=('ever_default', 'sum'),
    disposed_defaults=('disposed', 'sum'),
    median_credit_score=('credit_score', 'median'),
    median_original_upb=('original_upb', 'median'),
    last_obs_period=('last_period', 'max'),        # data extract horizon (YYYYMM)
    pct_seasoned_12m=('_seasoned_12m', 'mean'),     # share observed >= 12 months
)
dq['default_rate'] = (dq['defaults'] / dq['loans']).round(4)
dq['pct_seasoned_12m'] = dq['pct_seasoned_12m'].round(3)
dq = dq.reset_index()
df.drop(columns='_seasoned_12m', inplace=True)

In [5]:
# Save the one results table for this notebook.
from src.output import save_csv
save_csv(dq, 'outputs/tables/00_data_quality.csv')
dq

,vintage_year,loans,defaults,disposed_defaults,median_credit_score,median_original_upb,last_obs_period,pct_seasoned_12m,default_rate
0,2006,50000,6027,4066,729.0,158000.0,202509,0.938,0.1205
1,2007,50000,6870,4479,732.0,160000.0,202509,0.926,0.1374
2,2008,50000,3677,2134,753.0,183000.0,202509,0.860,0.0735
3,2009,50000,1137,543,771.0,188000.0,202509,0.958,0.0227
4,2010,50000,1095,496,772.0,178000.0,202509,0.944,0.0219
5,2011,50000,1019,379,773.0,178000.0,202509,0.913,0.0204
6,2012,50000,1115,379,773.0,181000.0,202509,0.953,0.0223
7,2013,50000,1175,305,762.0,176000.0,202509,0.970,0.0235
8,2014,50000,1188,202,757.0,184000.0,202509,0.931,0.0238
9,2015,50000,1209,136,760.0,200000.0,202509,0.949,0.0242


**Reading the table:** each vintage is a 50,000-loan random sample. The
`default_rate` column is the share of loans that ever hit serious default. The
crisis (2007-09) and COVID (2020) years stand well above the calm expansion years
-- a full good-and-bad cycle, which is what the long-run regulatory calibration needs.

**Seasoning (R3-D2).** `last_obs_period` shows the panel is observed through **2025-09**
for every vintage, and `pct_seasoned_12m` is the share of each vintage seen for >=12
months. Even 2022 is ~96% seasoned, so the one-year PD window is fully observable for
all 17 vintages -- the sub-12-month loans are **early payoffs, not right-censoring**, so
**no vintage is excluded** from the long-run average. (Recent-vintage *LGD* workouts can
still be incomplete; that is handled by the incomplete-workout sensitivity in nb 04.)